# Linear static analysis

This page puts the beams and the plate from {doc}`Setting up the domains <domains>`
together with the loads from {doc}`Specifying boundary conditions <boundary_conditions>`.
To keep the page self-contained, the following cell sets them up again:

In [1]:
import numpy as np

from sigmaepsilon.solid.fourier import (
    NavierBeam,
    NavierPlate,
    LoadGroup,
    PointLoad,
    LineLoad,
    RectangleLoad,
)

# Euler-Bernoulli and Timoshenko beams
beam_length = 1000.0
width, height = 20.0, 80.0
young_modulus, poisson_ratio = 210000.0, 0.25
bending_stiffness = young_modulus * width * height**3 / 12
shear_stiffness = (5 / 6) * width * height * young_modulus / (2 * (1 + poisson_ratio))

bernoulli_beam = NavierBeam(beam_length, 100, EI=bending_stiffness)
timoshenko_beam = NavierBeam(beam_length, 100, EI=bending_stiffness, GA=shear_stiffness)

# Kirchhoff-Love plate
length_X, length_Y = (600.0, 800.0)
thickness = 25.0
young_modulus, poisson_ratio = 2890.0, 0.2
D_factor = young_modulus * thickness**3 / (12 * (1 - poisson_ratio**2))
plate_bending_stiffness = D_factor * np.array(
    [[1, poisson_ratio, 0], [poisson_ratio, 1, 0], [0, 0, (1 - poisson_ratio) / 2]]
)

kirchhoff_plate = NavierPlate((length_X, length_Y), (20, 20), D=plate_bending_stiffness)

# loads
beam_loads = LoadGroup(
    concentrated=LoadGroup(
        LC1=PointLoad(beam_length / 2, [1.0, 0.0]),
        LC2=PointLoad(beam_length / 2, [0.0, 1.0]),
    ),
    distributed=LoadGroup(
        LC3=LineLoad([0, beam_length], [1.0, 0.0]),
        LC4=LineLoad([beam_length / 2, beam_length], [0.0, 1.0]),
    ),
)

plate_loads = LoadGroup(
    LC1=PointLoad([length_X / 2, length_Y / 2], [-1.0, 0.0, 0.0]),
    LC2=RectangleLoad([[0, 0], [length_X, length_Y]], [-0.1, 0, 0]),
)

Besides the loads, you also have to provide the points of evaluation where you want the results to be calculated.
To perform a linear static analysis, call the `linear_static_analysis` instance method.

In [2]:
import numpy as np

# the locations of the points where the solution is calculated
beam_points = np.linspace(0, beam_length, 500)

bernoulli_solution = bernoulli_beam.linear_static_analysis(beam_points, beam_loads)
timoshenko_solution = timoshenko_beam.linear_static_analysis(beam_points, beam_loads)

For plates, we can calculate the locations of the points of evaluation using `numpy.meshgrid`.


In [3]:
x = np.linspace(0, length_X, 30)
y = np.linspace(0, length_Y, 40)
xv, yv = np.meshgrid(x, y)
plate_points = np.stack((xv.flatten(), yv.flatten()), axis=1)

kirchhoff_results = kirchhoff_plate.linear_static_analysis(plate_points, plate_loads)

## Understanding the results


The `linear_static_analysis` method in both cases returns a dictionary with an identical layout as the loads. So for 
instance, if a load case was accessible as `beam_loads["concentrated", "LC1"]`, then the corresponding 
results are accessible as in `bernoulli_solution["concentrated", "LC1"]`. 
Each of these entries is an instance of {class}`~sigmaepsilon.solid.fourier.result.BeamLoadCaseResultLinStat`.

In [4]:
type(bernoulli_solution)

sigmaepsilon.deepdict.deepdict.DeepDict

In [5]:
type(bernoulli_solution["concentrated", "LC1"])

sigmaepsilon.solid.fourier.result.BeamLoadCaseResultLinStat

### The results as a `numpy.ndarray`

The out-of-the-box way of representing data is via NumPy arrays. This means a 2d NumPy array for each load case, where the first axis goes along the points of evaluation and the second along the components (strains, internal forces, etc.). This array is accessible through the {attr}`~sigmaepsilon.solid.fourier.result.LoadCaseResultLinStat.data` and {attr}`~sigmaepsilon.solid.fourier.result.LoadCaseResultLinStat.values` properties.

In [6]:
# 6 results components for 500 points
bernoulli_solution["concentrated", "LC1"].data.shape

(500, 6)

It is not very convenient to memorize the indices of the components. If you want a reminder, you can use the `components` property.

In [7]:
bernoulli_solution["concentrated", "LC1"].components

['UY', 'ROTZ', 'CZ', 'EXY', 'MZ', 'SY']

For more details, see the documentation of the classes {class}`~sigmaepsilon.solid.fourier.result.BeamLoadCaseResultLinStat` and {class}`~sigmaepsilon.solid.fourier.result.PlateLoadCaseResultLinStat`.

### The results as an `xarray.DataArray`

You can turn the results into an instance of {class}`xarray.DataArray` by calling the `to_xarray` method of the instance (this requires `xarray` to be installed):

In [8]:
xarr = bernoulli_solution["concentrated", "LC1"].to_xarray()

If you are familiar with `xarray`, you know that it is basically the same as a `NumPy` array, except that the values are also accessible using labels, besides the usual index-based nature of `NumPy`. The `indexes` attribute tells you about the available options.


In [9]:
xarr.indexes

Indexes:
    index      Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       490, 491, 492, 493, 494, 495, 496, 497, 498, 499],
      dtype='int64', name='index', length=500)
    component  Index(['UY', 'ROTZ', 'CZ', 'EXY', 'MZ', 'SY'], dtype='str', name='component')
    x          Index([               0.0,  2.004008016032064,  4.008016032064128,
        6.012024048096192,  8.016032064128256, 10.020040080160319,
       12.024048096192384, 14.028056112224448,  16.03206412825651,
       18.036072144288575,
       ...
        981.9639278557113,  983.9679358717434,  985.9719438877754,
        987.9759519038075,  989.9799599198395,  991.9839679358716,
        993.9879759519036,  995.9919839679358,  997.9959919839679,
                   1000.0],
      dtype='float64', name='x', length=500)

You can see that the array is two dimensional and the two axes are called 'index' and 'component'. Here 'index' refers to the integer index of the point of evaluation, while 'component' refers to result component associated with a beam.

Of course, for Euler-Bernoulli beams, the shear strains and shear forces are zero. To access the vertical displacement of the 10th evaluation point, use the `loc` method of the DataArray like this:


In [10]:
xarr.loc[9, "UY"]

<xarray.DataArray 'values' ()> Size: 8B
array(6.28775469e-06)
Coordinates:
    index      int64 8B 9
    component  <U4 16B 'UY'
    x          float64 8B 18.04

The same query with the `sel` method:

In [11]:
xarr.sel(index=9, component="UY")

<xarray.DataArray 'values' ()> Size: 8B
array(6.28775469e-06)
Coordinates:
    index      int64 8B 9
    component  <U4 16B 'UY'
    x          float64 8B 18.04

```{important}
**Like** `pandas`, **label based indexing in** `xarray` **is inclusive of both the start and stop bounds.** For instance, to access the same result component for the first 10 points, you have to slice the data with `[:9, 'UY']` instead of `[:10, 'UY']`:
```

In [12]:
xarr.loc[:9, "UY"]

<xarray.DataArray 'values' (index: 10)> Size: 80B
array([0.00000000e+00, 6.98938448e-07, 1.39785457e-06, 2.09672599e-06,
       2.79553025e-06, 3.49424481e-06, 4.19284710e-06, 4.89131451e-06,
       5.58962453e-06, 6.28775469e-06])
Coordinates:
  * index      (index) int64 80B 0 1 2 3 4 5 6 7 8 9
  * x          (index) float64 80B 0.0 2.004 4.008 6.012 ... 14.03 16.03 18.04
    component  <U4 16B 'UY'

You can also use the `sel` method to to the same thing:

In [13]:
xarr.sel(index=slice(9), component="UY")

<xarray.DataArray 'values' (index: 10)> Size: 80B
array([0.00000000e+00, 6.98938448e-07, 1.39785457e-06, 2.09672599e-06,
       2.79553025e-06, 3.49424481e-06, 4.19284710e-06, 4.89131451e-06,
       5.58962453e-06, 6.28775469e-06])
Coordinates:
  * index      (index) int64 80B 0 1 2 3 4 5 6 7 8 9
  * x          (index) float64 80B 0.0 2.004 4.008 6.012 ... 14.03 16.03 18.04
    component  <U4 16B 'UY'

### The results as a `pandas.DataFrame`

You can also turn the results into a {class}`pandas.DataFrame`:

In [14]:
df = bernoulli_solution["concentrated", "LC1"].to_pandas()
df

,UY,ROTZ,CZ,EXY,MZ,SY
index,,,,,,
0,0.000000e+00,3.487721e-07,0.000000e+00,0.0,0.000000e+00,0.496817
1,6.989384e-07,3.487666e-07,-5.558252e-12,0.0,-9.960388e-01,0.497428
2,1.397855e-06,3.487499e-07,-1.112927e-11,0.0,-1.994365e+00,0.499024
3,2.096726e-06,3.487220e-07,-1.672093e-11,0.0,-2.996390e+00,0.500996
4,2.795530e-06,3.486828e-07,-2.233318e-11,0.0,-4.002106e+00,0.502586
...,...,...,...,...,...,...
495,2.795530e-06,-3.486828e-07,-2.233318e-11,0.0,-4.002106e+00,-0.502586
496,2.096726e-06,-3.487220e-07,-1.672093e-11,0.0,-2.996390e+00,-0.500996
497,1.397855e-06,-3.487499e-07,-1.112927e-11,0.0,-1.994365e+00,-0.499024


Then you can query the object quite similarly to a `DataArray`.

In [15]:
df.loc[9, 'UY']

np.float64(6.2877546906128156e-06)

In [16]:
df.loc[:9, 'UY']

index
0    0.000000e+00
1    6.989384e-07
2    1.397855e-06
3    2.096726e-06
4    2.795530e-06
5    3.494245e-06
6    4.192847e-06
7    4.891315e-06
8    5.589625e-06
9    6.287755e-06
Name: UY, dtype: float64

### Reading the results by location

Rather than remembering which integer index belongs to which point, you can work with the coordinates of the points directly. Every result knows the points it was evaluated at:

In [17]:
kirchhoff_result = kirchhoff_results["LC1"]
kirchhoff_result.points[:3]

array([[ 0.        ,  0.        ],
       [20.68965517,  0.        ],
       [41.37931034,  0.        ]])

Components can be accessed by their names, which gives you a NumPy array along the points:

In [18]:
kirchhoff_result["UZ"][:3]

array([0., 0., 0.])

Since the solution is a Fourier series, it can be evaluated exactly at any point of the domain, there is no need for interpolation. The {func}`~sigmaepsilon.solid.fourier.result.LoadCaseResultLinStat.at` method evaluates the solution of the load case at one or more points, and returns the results in the same form:

In [19]:
kirchhoff_result.at([length_X / 3, length_Y / 4])["UZ"]

array([-0.00065684])

In [20]:
kirchhoff_result.at([[100.0, 200.0], [300.0, 400.0]]).to_pandas(coords=True)[["x", "y", "UZ", "MX"]]

,x,y,UZ,MX
index,,,,
0,100.0,200.0,-0.000370,-0.034299
1,300.0,400.0,-0.001329,-0.352957


The coordinates are also attached to the `xarray` representations. For beams, the results can be selected by location if the points are monotonic:

In [21]:
xarr.sel(x=beam_length / 3, method="nearest").sel(component="UY")

<xarray.DataArray 'values' ()> Size: 8B
array(9.89044757e-05)
Coordinates:
    index      int64 8B 166
    component  <U4 16B 'UY'
    x          float64 8B 332.7

For plates, if the points of evaluation form a full rectangular grid, you can ask for a grid-shaped array with the dimensions `x`, `y` and `component`:

In [22]:
kirchhoff_grid = kirchhoff_result.to_xarray(grid=True)
kirchhoff_grid.sel(x=length_X / 2, y=length_Y / 2, method="nearest").sel(component="UZ")

<xarray.DataArray 'values' ()> Size: 8B
array(-0.00132145)
Coordinates:
    x          float64 8B 289.7
    y          float64 8B 389.7
    component  <U4 16B 'UZ'

```{note}
The `coords` argument of `to_pandas` adds the coordinates of the points as columns to the data frame. It is off by default, so that the columns are the components of the results only.
```